# 🚀 LightGBM Baseline Model
**ROGII — Wellbore Geology Prediction**

> Train a LightGBM regression baseline using GroupKFold (by well) cross-validation.
> Includes OOF RMSE tracking, feature importance, and submission generation.

---
**Author:** Md Ashraf | M.Sc (Tech) Applied Geophysics, IIT (ISM) Dhanbad

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

from src import preprocessing, feature_engineering, validation, model_training, inference, plotting

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

In [ ]:
# Load configs
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../configs/lgbm_config.yaml') as f:
    lgbm_cfg = yaml.safe_load(f)

TARGET     = cfg['data']['target_column']
WELL_COL   = cfg['data']['well_id_column']
DEPTH_COL  = cfg['data']['depth_column']
PROC_DIR   = '../' + cfg['paths']['processed_dir']
RAW_DIR    = '../' + cfg['paths']['raw_dir']
MODELS_DIR = '../' + cfg['paths']['models_dir']
PLOTS_DIR  = '../' + cfg['paths']['plots_dir']
SUBS_DIR   = '../' + cfg['paths']['submissions_dir']
N_FOLDS    = cfg['validation']['n_folds']
SEED       = cfg['seed']

for d in [MODELS_DIR, PLOTS_DIR, SUBS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Config loaded ✅')

## 1. Load Processed Data
*(Runs feature engineering inline if processed files are not found)*

In [ ]:
proc_train = Path(f'{PROC_DIR}/train_features.parquet')
proc_test  = Path(f'{PROC_DIR}/test_features.parquet')

if proc_train.exists() and proc_test.exists():
    train = pd.read_parquet(proc_train)
    test  = pd.read_parquet(proc_test)
    with open(f'{PROC_DIR}/feature_cols.json') as f:
        feature_cols = json.load(f)
    print(f'Loaded processed data — Train: {train.shape} | Test: {test.shape}')
else:
    print('Processed data not found — running feature engineering inline...')
    # --- Inline demo data & feature engineering ---
    np.random.seed(SEED)
    n_wells, n_depth = 8, 500
    rows = []
    for w in range(n_wells):
        for d in range(n_depth):
            rows.append({
                'well_id': f'WELL_{w:02d}',
                'md': 2000 + d * 2,
                'tvd': 1800 + d * 0.5 + np.random.randn() * 5,
                'inclination': 85 + np.random.randn() * 2,
                'azimuth': 135 + np.random.randn() * 3,
                'rop': np.random.lognormal(2.5, 0.4),
                'wob': np.random.lognormal(3.2, 0.3),
                'rpm': np.random.normal(120, 15),
                'torque': np.random.lognormal(4.0, 0.5),
                'flow_rate': np.random.normal(400, 30),
                'ecd': np.random.normal(1.35, 0.05),
                'gr': np.random.lognormal(4.2, 0.5),
                'resistivity': np.random.lognormal(2.0, 1.0),
                'neutron': np.random.normal(0.25, 0.05),
                'density': np.random.normal(2.4, 0.1),
                'sonic': np.random.normal(90, 10),
                'formation': np.random.uniform(0, 10),
            })
    train = pd.DataFrame(rows).sort_values([WELL_COL, DEPTH_COL]).reset_index(drop=True)
    test  = train.sample(frac=0.15, random_state=SEED).drop(columns=['formation']).reset_index(drop=True)

    feat_cfg = cfg['features']
    drilling_cols = [c for c in feat_cfg.get('drilling_features', []) if c in train.columns]
    petro_cols    = [c for c in feat_cfg.get('petrophysical_features', []) if c in train.columns]
    all_base      = drilling_cols + petro_cols

    for df_name, df in [('train', train), ('test', test)]:
        df = feature_engineering.add_rolling_features(df, all_base, [3, 5, 10], group_col=WELL_COL)
        df = feature_engineering.add_lag_features(df, all_base, [1, 2, 3], group_col=WELL_COL)
        df = feature_engineering.add_depth_features(df, depth_col=DEPTH_COL, well_col=WELL_COL)
        df = feature_engineering.add_drilling_features(df)
        df = feature_engineering.add_petrophysical_features(df)
        df = feature_engineering.add_directional_features(df)
        if df_name == 'train':
            train = df
        else:
            test = df

    exclude = [TARGET, WELL_COL, 'id']
    feature_cols = [c for c in train.columns if c not in exclude
                    and train[c].dtype in [np.float32, np.float64, np.int32, np.int64]]
    med = train[feature_cols].median()
    train[feature_cols] = train[feature_cols].fillna(med)
    test[feature_cols]  = test[feature_cols].fillna(med)

    print(f'Feature engineering complete — Train: {train.shape} | Features: {len(feature_cols)}')

## 2. GroupKFold Splits (by Well)

In [ ]:
cv_splits = validation.get_cv_splits(
    train,
    strategy=cfg['validation']['strategy'],
    n_folds=N_FOLDS,
    group_col=WELL_COL,
)

print(f'Number of folds: {len(cv_splits)}')
print(f'Unique wells: {train[WELL_COL].nunique()}')

# Show well distribution per fold
for fold_idx, (tr_idx, vl_idx) in enumerate(cv_splits, start=1):
    val_wells = train.iloc[vl_idx][WELL_COL].unique()
    print(f'Fold {fold_idx}: Val wells = {list(val_wells)} | Val rows = {len(vl_idx)}')

## 3. Train LightGBM with Cross-Validation

In [ ]:
lgbm_params = lgbm_cfg['params']

models, oof_preds, fold_scores = model_training.train_lightgbm_cv(
    df=train,
    feature_cols=feature_cols,
    target_col=TARGET,
    cv_splits=cv_splits,
    params=lgbm_params,
    early_stopping_rounds=lgbm_cfg['early_stopping']['rounds'],
    save_dir=MODELS_DIR,
)

validation.print_cv_summary(fold_scores)

## 4. OOF Performance Metrics

In [ ]:
y_true = train[TARGET].values
metrics = validation.evaluate(y_true, oof_preds)

print('=== OOF Evaluation Metrics ===')
for k, v in metrics.items():
    print(f'  {k.upper()}: {v:.6f}')

## 5. OOF Diagnostics Plot

In [ ]:
fig = plotting.plot_oof_diagnostics(
    y_true, oof_preds,
    title=f'LightGBM OOF Diagnostics  |  RMSE={metrics["rmse"]:.5f}',
    save_path=f'{PLOTS_DIR}/lgbm_oof_diagnostics.png',
)
plt.show()

## 6. CV Score Plot

In [ ]:
fig = plotting.plot_cv_scores(
    fold_scores,
    model_name='LightGBM Baseline',
    save_path=f'{PLOTS_DIR}/lgbm_cv_scores.png',
)
plt.show()

## 7. Feature Importance

In [ ]:
# Average importance across folds
import numpy as np

importances = np.zeros(len(feature_cols))
for model in models:
    importances += model.feature_importances_
importances /= len(models)

fig = plotting.plot_feature_importance(
    feature_cols, importances, top_n=30,
    title='LightGBM Feature Importance (avg across folds)',
    save_path=f'{PLOTS_DIR}/lgbm_feature_importance.png',
)
plt.show()

## 8. Generate Test Predictions & Submission

In [ ]:
test_preds = inference.predict(models, test[feature_cols], aggregation='mean')

print(f'Test predictions — shape: {test_preds.shape}')
print(f'  Min: {test_preds.min():.4f} | Max: {test_preds.max():.4f} | Mean: {test_preds.mean():.4f}')

In [ ]:
id_col = 'id' if 'id' in test.columns else test.columns[0]

sub = inference.make_submission(
    test_df=test if id_col in test.columns else test.assign(id=range(len(test))),
    predictions=test_preds,
    id_col=id_col,
    target_col=TARGET,
    output_path=f'{SUBS_DIR}/submission_lgbm_baseline.csv',
)

display(sub.head(10))
print(f'\nSubmission saved ✅')

---
## ✅ Summary

Run the cell below to display the final metrics table after training:

```python
print(f"| OOF RMSE | {metrics["rmse"]:.5f} |")
print(f"| OOF MAE  | {metrics["mae"]:.5f} |")
print(f"| OOF R²   | {metrics["r2"]:.5f} |")
print(f"| CV Folds | {N_FOLDS} GroupKFold (by well) |")
```

**Next steps:**
- Run Optuna for hyperparameter tuning (`model_training.run_optuna_lgbm()`)
- Add XGBoost / CatBoost folds for blending
- Explore deep learning models (LSTM / Transformer) in `src/deep_learning.py`